sasho beshe tuk

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import HeatMap
import matplotlib.pyplot as plt

In [7]:
df_sightings = pd.read_csv('./Phalacrocorax carbo.csv',sep=',', low_memory=False)
df_habitats = pd.read_csv('./habitats_cbs_2022.csv', sep=',', low_memory=False)

df = df_sightings.merge(
    df_habitats,
    on=["decimalLatitude", "decimalLongitude"],
    how="inner",
)

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12558756 entries, 0 to 12558755
Data columns (total 15 columns):
 #   Column                     Dtype  
---  ------                     -----  
 0   decimalLatitude            float64
 1   decimalLongitude           float64
 2   eventDate                  str    
 3   total_observations         float64
 4   speciesgroup_observations  int64  
 5   Phalacrocorax carbo        str    
 6   agricultural               float64
 7   built                      float64
 8   coast                      float64
 9   forest                     float64
 10  other                      float64
 11  sand/heather               float64
 12  water                      float64
 13  wetland                    float64
 14  main_habitat               str    
dtypes: float64(11), int64(1), str(3)
memory usage: 1.4 GB


In [9]:
pd.to_datetime(df['eventDate'])
cols = [
    "total_observations",
    "speciesgroup_observations",
    "Phalacrocorax carbo"
]

df[cols] = df[cols].apply(pd.to_numeric, errors="coerce").fillna(0)

In [10]:
df.to_csv('bird_and_habitats.csv', index=False)

In [12]:
print(df.dtypes)

decimalLatitude              float64
decimalLongitude             float64
eventDate                        str
total_observations           float64
speciesgroup_observations      int64
Phalacrocorax carbo            int64
agricultural                 float64
built                        float64
coast                        float64
forest                       float64
other                        float64
sand/heather                 float64
water                        float64
wetland                      float64
main_habitat                     str
dtype: object


In [13]:
# --- convert to numeric ---
cols = [
    "total_observations",
    "speciesgroup_observations",
    "Phalacrocorax carbo"
]

df[cols] = df[cols].apply(pd.to_numeric, errors="coerce").fillna(0)

# --- keep valid coordinates ---
df_clean = df.dropna(subset=["decimalLatitude", "decimalLongitude"])

# --- effort adjustment (important step) ---
df_clean["adjusted_weight"] = (
    df_clean["Phalacrocorax carbo"] /
    (df_clean["total_observations"] + df_clean["speciesgroup_observations"] + 1)
)

In [14]:
grouped = df_clean.groupby(
    ["decimalLatitude", "decimalLongitude"],
    as_index=False
)["adjusted_weight"].sum()

grouped = grouped[grouped["adjusted_weight"] > 5]

In [15]:
heat_data = grouped.values.tolist()

m = folium.Map(location=[52.1, 5.3], zoom_start=7)

HeatMap(
    heat_data,
    radius=20,
    blur=15,
    min_opacity=0.1,
    max_zoom=12
).add_to(m)

m.save("heatmap.html")

In [20]:
# Check for missing data
print("Missing values per column:")
print(df_clean.isnull().sum())

# Check distribution
print("\nTarget Variable Summary Statistics:")
print(df_clean['Phalacrocorax carbo'].describe())

# Check if any records are out of the Netherlands
out_of_bounds = df_clean[
    (df_clean['decimalLatitude'] < 50.5) | (df_clean['decimalLatitude'] > 53.7) |
    (df_clean['decimalLongitude'] < 3.2) | (df_clean['decimalLongitude'] > 7.2)
]
print(f"\nNumber of spatial records outside the Netherlands: {len(out_of_bounds)}")

Missing values per column:
decimalLatitude              0
decimalLongitude             0
eventDate                    0
total_observations           0
speciesgroup_observations    0
Phalacrocorax carbo          0
agricultural                 0
built                        0
coast                        0
forest                       0
other                        0
sand/heather                 0
water                        0
wetland                      0
main_habitat                 0
adjusted_weight              0
dtype: int64

Target Variable Summary Statistics:
count    1.255876e+07
mean     5.143431e-02
std      3.364980e-01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.090000e+02
Name: Phalacrocorax carbo, dtype: float64

Number of spatial records outside the Netherlands: 0


## Data Quality
- There are zero missing values in the dataset across all columns and that confirms that all lat/long coordinates are well bounded inside the Netherlands (50.5 to 53.7 N, 3.2 to 7.2E)

In [24]:
from sklearn.preprocessing import LabelEncoder
import numpy as np

# Feature engineering
df_clean['eventDate'] = pd.to_datetime(df_clean['eventDate'])
df_clean['month'] = df_clean['eventDate'].dt.month
df_clean['day_of_year'] = df_clean['eventDate'].dt.dayofyear
df_clean['is_breeding_season'] = df_clean['month'].isin([3, 4, 5, 6]).astype(int) # March-June

# Fill missing habitat with 0
habitat_cols = ['agricultural', 'built', 'coast', 'forest', 'other', 'sand/heather', 'water', 'wetland']
df_clean[habitat_cols] = df_clean[habitat_cols].fillna(0)

# Convert 'main_habitat' string to integers
le = LabelEncoder()
df_clean['main_habitat_encoded'] = le.fit_transform(df_clean['main_habitat'].astype(str))

# Drop columns we won't pass directly into the model
features = [
    'decimalLatitude', 'decimalLongitude', 'month', 'day_of_year', 'is_breeding_season',
    'total_observations', 'speciesgroup_observations',
    'agricultural', 'built', 'coast', 'forest', 'other', 'sand/heather', 'water', 'wetland',
    'main_habitat_encoded'
]

X = df_clean[features]
y = df_clean['adjusted_weight'] # This is your engineered target variable!

print("Data shape ready for modeling:", X.shape)

Data shape ready for modeling: (12558756, 16)


## Data preparation
- Converted the timestamp into cyclical indicators (`month`, `day_of_year`) and engineered a domain-specific binary flag (`is_breeding_season`) based on the March-June window.
- Used a `LabelEncoder` to cast the text column `main_habitat` into a numerical integer mapping (`main_habitat_encoded`) and isolated a clean feature matrix `X` alongside the target variable `y`

## Choosing a model and metrics
### Random Forest
- The tree erxhitrecture can pick up non-linear environmental conjstraints (If near water AND it is breeding season, then bird density spikes)

### Metrics
Since the data is mostly zeros, standart metrics like Mean Absolute Error (MAE) or R2 are not a good choice. A model that predicts that exactly 0 everywhere will get a seemingly great error score but provide zero real utility. For this dataset we can use:

- *Mean Tweedie Deviance* - Measures goodness of fit for zero-heavy continuous distributions.
- *Root Mean Squared Log Error (RMSLE)* - Since the maximum counts spike to 109, log errors penalize massive underpredictions well without blowing up the metric due to variance.